# `NescienceRegressor` demo

This notebook showcases the new `NescienceRegressor` class for the revised `mnplib` architecture.

The regressor implements **minimum-nescience model selection**:

```python
fit candidate model on (X, y)
extract explicit artifacts:
    subset, predictions, model_string
compute nescience
select the candidate with minimum nescience
```

The class does **not** use a train/test split or cross-validation. The theory evaluates the model with respect to the available effective representation `(X, y)`. Excessive complexity is penalized internally through the nescience components, especially `surfeit` and `surplus`.

This demo covers:

- default candidate search;
- custom candidate lists;
- result tables;
- component plots;
- explanations;
- canonical model-string inspection;
- weight sensitivity;
- DataFrame feature-name preservation;
- strict unsupported-model behavior.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

from mnplib.regressor import NescienceRegressor
from pprint import pformat

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Helper functions

These helpers are only for notebook display. They are not part of the library.


In [ ]:
def plot_nescience(df, title):
    """Plot scalar nescience for candidate models."""
    df.set_index("candidate")["nescience"].plot(kind="bar", figsize=(11, 4))
    plt.ylabel("Nescience")
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_components(df, title):
    """Plot the four nescience components for candidate models."""
    component_cols = ["deficiency", "surplus", "inaccuracy", "surfeit"]
    df.set_index("candidate")[component_cols].plot(kind="bar", figsize=(12, 5))
    plt.ylabel("Component value")
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def print_section(title):
    """Print a visible separator."""
    print("\n" + title)
    print("=" * len(title))

## 3. Build a synthetic regression problem

The dataset has eight input features, but only three are informative. This is useful for seeing how different candidate models trade off accuracy, feature use, and description length.


In [ ]:
X, y = make_regression(
    n_samples=700,
    n_features=8,
    n_informative=3,
    noise=15.0,
    random_state=42,
)

feature_names = [f"x{i}" for i in range(X.shape[1])]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Fit the default `NescienceRegressor`

The default candidate set currently includes:

- `LinearRegression`
- several `Ridge` models;
- several `Lasso` models;
- several `ElasticNet` models;
- several `DecisionTreeRegressor` models.

All candidates are evaluated on the same representation `(X, y)`.


In [ ]:
reg = NescienceRegressor(
    random_state=42,
    verbose=1,
)

reg.fit(X, y)

## 5. Selected model and main diagnostics

In [ ]:
print("Best candidate:", reg.best_candidate_name_)
print("Selected estimator:", type(reg.model_).__name__)
print("Best nescience:", reg.nescience())
print("Native estimator score on full data:", reg.score(X, y))

print("\nComponents:")
for name, value in reg.components().items():
    print(f"{name:>10}: {value:.6f}")

## 6. Candidate comparison table

`results_dataframe()` gives a compact table with the selected model first because rows are sorted by ascending nescience.


In [ ]:
results = reg.results_dataframe()
results

In [ ]:
plot_nescience(results, "Default candidates: scalar nescience")
plot_components(results, "Default candidates: nescience components")

## 7. Native predictive score versus nescience

The native estimator score for regressors is usually \(R^2\), while nescience is a different objective. A candidate may score very well predictively but still have higher nescience because it uses an unnecessarily complex or redundant description.


In [ ]:
comparison = results[
    [
        "candidate",
        "model_type",
        "native_estimator_score",
        "nescience",
        "inaccuracy",
        "surfeit",
        "n_selected_features",
        "description_length",
    ]
].copy()

comparison.sort_values("native_estimator_score", ascending=False)

In [ ]:
comparison.set_index("candidate")[["native_estimator_score", "nescience"]].plot(
    kind="bar",
    figsize=(11, 4),
)
plt.title("Native estimator score versus nescience")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Explanation of the selected model

`analysis()` returns the selected candidate's metrics, effective features, and reported hyperparameters. `pformat()` presents the dictionary as a formatted numerical dictionary. The candidate-evaluation R-squared was recorded on training data, not a held-out set.


In [ ]:
explanation = reg.analysis()

print(pformat(explanation))

## 9. Inspect the canonical model string

The selected model is serialized into the canonical model-description schema used by the new scikit-learn adapter layer.


In [ ]:
model_string = reg.model_description()["model_string"]

print(model_string[:2500])
print("\nDescription length in bytes:", len(model_string.encode("utf-8")))

## 10. Custom candidate set

For demos or controlled experiments, it is often better to pass an explicit candidate list.


In [ ]:
custom_candidates = [
    ("linear", LinearRegression()),
    ("ridge_1", Ridge(alpha=1.0)),
    ("lasso_001", Lasso(alpha=0.01, max_iter=10000)),
    ("elastic_net_001", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)),
    ("tree_depth_2", DecisionTreeRegressor(max_depth=2, random_state=42)),
    ("tree_depth_4", DecisionTreeRegressor(max_depth=4, random_state=42)),
]

custom_reg = NescienceRegressor(
    random_state=42
)

custom_reg.fit(X, y)

custom_results = custom_reg.results_dataframe()
custom_results

In [ ]:
plot_nescience(custom_results, "Custom candidates: scalar nescience")
plot_components(custom_results, "Custom candidates: nescience components")

## 11. DataFrame feature names

If `X` is a pandas DataFrame, the canonical model strings preserve column names.


In [ ]:
X_df = pd.DataFrame(X, columns=[f"feature_{j}" for j in range(X.shape[1])])

df_reg = NescienceRegressor(
    random_state=42
)

df_reg.fit(X_df, y)

print("Feature names stored in the regressor:")
print(list(df_reg.feature_names_in_))

print("\nCanonical model string excerpt:")
print(df_reg.model_description()["model_string"][:1500])

## 12. Summary

The new `NescienceRegressor` is a small model-selection orchestrator:

```python
NescienceRegressor
    -> fits candidate regressors on (X, y)
    -> extracts canonical artifacts with mnplib.models
    -> computes nescience with Nescience
    -> selects the candidate with minimum nescience
```

Its most useful outputs are:

```python
reg.best_candidate_name_
reg.best_nescience_
reg.components()
reg.analysis()
print(pformat(reg.analysis()))
reg.results_dataframe()
reg.model_description()["model_string"]
```

This keeps the model-selection logic separate from the metric classes and from the model serializers.
